# 03 - Validation, uncertainty and temporal backtesting


A model-risk review should ask more than whether a point estimate looks good. This notebook checks discrimination, calibration, population stability, bootstrap uncertainty, challenger-model separation, and rolling out-of-time performance.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.data import generate_portfolio, FEATURES, TARGET
from src.model import build_logistic_model, build_challenger_model
from src.validation import evaluate_model, calibration_table, population_stability_index, bootstrap_confidence_intervals, bootstrap_auc_difference
from run_analysis import rolling_backtest
df = generate_portfolio()
train = df[df.observation_year <= 2024]
test = df[df.observation_year == 2025]
primary = build_logistic_model().fit(train[FEATURES], train[TARGET])
challenger = build_challenger_model().fit(train[FEATURES], train[TARGET])
train_p = primary.predict_proba(train[FEATURES])[:, 1]
test_p = primary.predict_proba(test[FEATURES])[:, 1]
challenger_p = challenger.predict_proba(test[FEATURES])[:, 1]
metrics = evaluate_model(test[TARGET], test_p)
metrics['score_psi'] = population_stability_index(train_p, test_p)
metrics


{'roc_auc': 0.7633930227580794,
 'ks_statistic': 0.37824831751920285,
 'brier_score': 0.09796080207657945,
 'mean_predicted_pd': 0.10854776866479562,
 'observed_default_rate': 0.12809647979139505,
 'score_psi': 0.0033466560477962352}

In [2]:
bootstrap_confidence_intervals(test[TARGET], test_p, n_bootstrap=500).round(4)


,metric,estimate,ci_lower,ci_upper,bootstrap_replicates
0,roc_auc,0.7634,0.7388,0.7886,500
1,ks_statistic,0.3782,0.3465,0.4341,500
2,brier_score,0.0980,0.0902,0.1071,500


In [3]:
bootstrap_auc_difference(test[TARGET], test_p, challenger_p, n_bootstrap=500)


{'auc_difference': 0.06600366221968557,
 'ci_lower': 0.04484931086342031,
 'ci_upper': 0.08592045824302302,
 'bootstrap_replicates': 500}

In [4]:
calibration_table(test[TARGET].to_numpy(), test_p).round(4)


,borrowers,predicted_pd,observed_default_rate,calibration_gap
0,307,0.0189,0.0163,0.0026
1,307,0.0312,0.0391,-0.0079
2,307,0.0422,0.0391,0.0031
3,306,0.0546,0.0392,0.0154
4,307,0.0688,0.1107,-0.0420
5,307,0.0863,0.1173,-0.0310
6,306,0.1092,0.1209,-0.0117
7,307,0.1397,0.1368,0.0029
8,307,0.1914,0.2313,-0.0398
9,307,0.3431,0.4300,-0.0869


In [5]:
rolling_backtest(df).round(4)


,validation_year,development_rows,validation_rows,roc_auc,ks_statistic,brier_score,mean_predicted_pd,observed_default_rate
0,2023,2668,2999,0.7282,0.3346,0.0858,0.1106,0.1040
1,2024,5667,3265,0.7410,0.3681,0.0892,0.1076,0.1093
2,2025,8932,3068,0.7634,0.3782,0.0980,0.1085,0.1281
